# model_00 - deneme 2 . KLASIK TRANSFORMER (referans kol)

Kullanici istegi, 16 Eylul: *"model 00 kuralim. Bu model her seyden
bagimsiz, senden istedigim standart kendi bilginle kuracagin 8 katmanli,
loop vs olmayan. Klasik LLM transformer yapisi olsun."*

Onceden kayit `belge/onkayit/model_00.md`.

### NE YOK

```
DONGU YOK            l=8 AYRI katman, agirlik paylasimi YOK (dongu=1)
Phi DARBOGAZI YOK    dar_alfa=0, dar_kapi=False -> hic kurulmuyor
MASKE YOK            mask_poz=None, mask_blok=()
YARDIMCI KAYIP YOK   kopru_kayip=0
MIRAS YOK            ModelSade, model_a.Model'i GENISLETMIYOR
```

`model_b` ailesi `M.Model`i miras aliyordu; bu almiyor. Amac, projenin
birikmis tercihlerinden (ogrenilmis pozisyon gommesi, GELU + 4d MLP)
**bagimsiz** bir referans kurmak.

### NE VAR - 2026'nin standart kucuk-LLM tarifi

```
decoder-only + causal maske
pre-norm + RMSNorm                bias YOK
RoPE                              ogrenilmis pozisyon gommesi YERINE
SwiGLU, d_ff = 8/3 * d = 704      GELU + 4d YERINE
bagli gomme (head.weight is emb.weight)
MHA, head_dim 64 (d=256, nh=4)    bu olcekte GQA/MQA anlamsiz
dropout 0
GPT-2 init: N(0, 0.02); artik yoluna yazanlar 1/sqrt(2L) olcekli
```

`wd=0.1` + cosine (`min_lr = lr/10`) CLAUDE.md **kural 4**'ten geliyor ve
zaten bu tarifin kendisi (nanoGPT, Pythia, Qwen2.5 SFT).

```
model_00    l=8, dongu=1  ->  8 katman-esdegeri
model_b15   l=4, dongu=2  ->  8 katman-esdegeri
                              AYNI hesap derinligi, ama 4 agirlik seti
```

### VERI model_b15 ILE BIREBIR AYNI - kasitli

*"Her seyden bagimsiz"* **mimari** icin gecerli, veri icin degil. Ayni
veriyi gormezse kol hicbir sey olcmez.

```
veri_00 (icerik veri_okul4, 1060 varlik)  ek_kip="tr"  bicim=3  t_len 17
ident_frac=0.2  wd=0.1  cosine  tam_kayip=True
olcme izi d6751004648c    <- model_b15 ile AYNI, KIYAS GECERLI
```

Veri modulu KENDI adiyla (`veri_00`) ama ICERIK ayni; `veri_00.IZ`
parmak izi bunu her cagrilista denetliyor.

Kilitte sinaniyor: 18 gorev alani + GRAF (sekiz anahtar) + egitim
havuzu birebir ayni; `model_00 <-> model_b15` farki **yalniz**
`['ad','betas','dar_alfa','dar_kapi','dff','dongu','l','ort_bas','veri_ad']`
-- yani MIMARI + standart tarif (betas, ort_bas) + veri modulunun ADI
(icerigi degil). Egitim havuzu **bit duzeyinde ayni** (297.548 x 17).

### !! PARAMETRE IKI KATI - bastan yazilir

```
dikkat  4 d^2     =   262.144      SwiGLU  3 d d_ff  =   540.672
katman basina     =   802.816      x8               = 6.422.528
gomme (bagli)     =    77.056      TOPLAM           = 6.503.936

model_b15                                           = 3.229.953
```

Ayni katman-esdegerinde ~2x parametre: dongu agirlik **paylasiyor**, bu
paylasmiyor. Kusur degil, **kolun tanimi**. Ama sonuc olumluysa
*"mimari mi, parametre mi"* **AYRILAMAZ** ve oyle yazilir; ayiran kol
`model_00s` (d=192, l=8 -> ~3,6M) olurdu.

### KARAR KURALI - son pencere

```
comp        HUKUM
>= 0.30     KLASIK MIMARI BILESIMI ACIYOR
            -> bizim mimari kollarimiz (dongu + Phi) YANLIS YOLMUS
0.10-0.30   kismi
<  0.10     mimari DEGIL -- arizanin kaynagi baska yerde

one/seen    EK OKUMA (hukum vermez)
>= model_b15   klasik tarif ezberi de en az o kadar iyi ogreniyor
<  model_b15   RoPE/SwiGLU bu dilde GERILEME

DOGRUSAL SONDA slot 0   EK OKUMA -- Physics 3.1'in iddiasi
```

> `ent`/`ood` raporlanir, HUKUM VERMEZ.
> **SAYISAL TAHMIN YAZILMIYOR** (kullanici karari, 15 Eylul).

### ON KOSUL

`one`/`seen` kapilari. Gecmezse `comp` **YORUMLANMAZ**.

### SANITY CHECK - gevsetilmedi

```
one >= 0.98   seen >= 0.95   comp >= 0.50   ent_yok_kisayol == 0
```

### Beklenen kilit ciktisi

```
model_00/test_00.py    121 gecti, 0 BOZUK

0) BAGIMSIZLIK  model_00/*.py AST ile taranir -- 11 dosyanin hicbiri
   model_a / model_b / veri_okul / pencere_a import ETMIYOR
1) MIMARI       dongu=1, l=8, Phi YOK, MASKE YOK, MIRAS YOK,
                8 blok AYRI agirlik, BAGLI GOMME, BIAS YOK,
                RoPE var ve OGRENILMIYOR, SwiGLU UC matris
                CAUSAL + RoPE POZISYON duyarliligi SINANDI
                govde() var -- DOGRUSAL SONDA yolu KOSULUYOR
2) VERI         veri_00 == veri_okul4: grafin SEKIZ anahtari,
                zincirler (104.940), SEMA, turetilebilir  BIREBIR
                IZ 3f6751c4ccd4 ve uc kor noktasi sinaniyor
3) MOTOR        taban_00 vs model_a: ESKI_VARSAYILAN, Ayar alanlari,
                EGITIM HAVUZU bit duzeyinde, olcme izi d6751004648c
4) TARIF        LOOKAHEAD KAPALI (ort_bas=0), betas (0.9,0.95),
                wd 0.1 + cosine, isinma 2000, lr 1e-3
5) PARAMETRE    6.503.936
6) SOZLESME     8 modul YALNIZ model_00/ ile import oluyor
```

---

**Sira:** 0 -> 1 -> 2 -> 3 -> 4 ile baslat. Kosu surerken **5 ve 6**.
Bitince **7** (`pencere_00`) ve **9** (`sor_00`).

## 0 — Model adı ve yollar

Değiştirilecek **tek satır** burada: `MODEL`. Depo yolu ve Drive yolu
ondan türer — ikisi de elle yazılmaz.

**Log adı burada üretilmez**, onu 4. hücre her başlatmada kendisi basar.
Böylece iki koşu aynı log adını alamaz, hücre sırası ne olursa olsun.

> **tekrar:** güvenli, her zaman · **GPU:** hayır · **yazar:** hayır

In [ ]:
MODEL = "model_00"            # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

## 1 — GPU var mı, boş mu

Hız ÖLÇÜLDÜ (15 Eylül, T4, t0). Ardışık iki ölçüm noktasının
FARKINDAN, ortanca değer:

```
model_a   l=4 dongu=2   32,3 ms/adim   20.000 adim  10,8 dk
model_a1  l=4 dongu=1   18,2 ms/adim   20.000 adim   6,1 dk
model_a2  l=8 dongu=1   34,1 ms/adim   20.000 adim  11,4 dk
```

Döngü adım başına **1,77×** pahalı — hesabın neredeyse tamamı
bloklarda. Sekiz AYRI katman (`model_a2`) ise döngülü sekiz
katman-eşdeğerinden yalnız **1,06×** pahalı: aynı iş, farklı ağırlıklar.

> ⚠️ **`egri`deki `sn` HER SÜRDÜRMEDE SIFIRLANIR.**
> `sn[20000]/20000` hesabı YANLIŞTIR — 15 Eylül'de tam böyle yanıldım:
> `model_a`nın 20.000. adımı 8.000'de başlamış bir oturumun içindeydi,
> 22,3 ms çıktı (gerçek 32,3) ve bu yanlış sayı dört deftere birden
> yazıldı. Adım başına süre **ardışık iki ölçüm noktasının
> farkından** hesaplanır.

CPU'ya düşerse aynı iş **saatler** sürer — arşivde adım başına 1,64 sn
ölçülmüştü, bu da 20.000 adım için ~9 saat eder. Burada durmak, 9 saat
sonra fark etmekten iyidir.

> **tekrar:** güvenli · **GPU:** sorar, kullanmaz · **yazar:** hayır

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

## 2 — Drive

Çıktı doğrudan Drive'a yazılır. `/content` runtime ölünce silinir, Drive silinmez.

`drive.mount` çalışmazsa `/content/drive/MyDrive/...` **sihirli bir yol
değildir** — sıradan bir klasördür ve `os.makedirs` onu geçici diskte
sessizce açar. Koşu biter, runtime ölür, her şey silinir.
`ismount` bunu ayırır.

> **tekrar:** güvenli · **GPU:** hayır · **yazar:** evet — yalnızca klasör
> açar (`<model>/log/`), hiçbir koşu dosyasına dokunmaz

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

## 3 — Kodu GitHub'dan çek

**Colab'da yama yok.** Depo silinip yeniden klonlanır, üzerine hiçbir şey
yazılmaz. Tek kaynak: GitHub.

Yereli düzeltip itmeyi unuttuysan burada görürsün — klon eski commit'i getirir.

Bu hucre ayrica ailenin **kilit testini** (`test_*.py`) kosturur:
taban `model_a` degismedi mi, kollarin dugme yapisi bozulmadi mi.
Duserse egitim **baslamaz** -- kayitli sonuclar o tabana dayaniyor.

> **tekrar:** ⚠️ **EĞİTİM KOŞARKEN ÇALIŞTIRMA.** İlk işi `rm -rf /content/kod`
> ve koşan süreç tam o klasörden import etmiş durumda. Koşu yokken güvenli.
> · **GPU:** hayır · **yazar:** `/content` (Drive'a **değil**)

In [ ]:
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
# !! model_00 ARANMAZ: kendi klasoru BELLI. Kullanici karari,
# 16 Eylul -- "model_00 diger hicbir model ile ayni seyi kullanmamali".
# Paylasilan sablon `deneme2/*/<MODEL>.py` glob'u yapiyordu; model_00
# artik o aramaya girmiyor.
_aday = glob.glob(f"{KOD}/deneme2/model_00/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- KILIT: test_00.py ---------------------------------------------------
# model_00 KENDI motoruna (taban_00.py) ve KENDI verisine (veri_00.py)
# sahip -- ikisi de model_a / veri_okul KOPYASI. `test_00.py` dort sey
# tutuyor: (0) BAGIMSIZLIK, model_00/*.py disariya import ETMIYOR;
# (1) MIMARI; (2) VERI, graf veri_okul4'unkiyle BIREBIR; (3) MOTOR,
# egitim havuzu ve olcme izi model_a ile AYNI.
# Duserse egitim BASLAMAMALI -- sayilar baska bir tabloda okunur.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
# !! UST KLASOR EKLENMIYOR: veri modulu de (veri_00.py) AILE icinde.
# model_00 `deneme2/` kokundeki hicbir seyi gormez.
M_00 = importlib.import_module(MODEL)
M = M_00
# `M_00.M` MOTOR (taban_00). Paylasilan sablonda `M` TABANI
# gosteriyordu (model_a); burada `M` kolun KENDISI, motor ise
# `M_00.M`. Miras denetimi taban sinifi ORADAN alir.
MOTOR = M_00.M
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
# BU KOLUN SARTLARI. Hicbiri DEVRALINMIYOR: `ayar_00.py` her alani
# `Ayar()` varsayilaninin uzerine tek tek, gerekcesiyle yaziyor.
# --- MIMARI: bu kolun TANIMI --------------------------------------
assert M.AYAR.dongu == 1,              "DONGU YOK"
assert M.AYAR.l == 8,                  "8 AYRI katman"
assert M.AYAR.dar_alfa == 0.0,         "Phi DARBOGAZI YOK"
assert M.AYAR.dar_kapi is False,       "ogrenilen GECIT YOK"
assert M.AYAR.dff == 704,              "SwiGLU d_ff = 8/3*d"
assert M.AYAR.d // M.AYAR.nh == 64,    "head_dim 64"
# KUSUR (16 Eylul, defterin ILK kosusu): burada `M.Model` yaziyordu
# ve `M` = model_00 modulu; onda `Model` YOK -> AttributeError.
# Sablondan devralinmisti, defter hic kosulmadigi icin gorulmedi.
assert not issubclass(M_00.ModelSade, MOTOR.Model), \
    "ModelSade taban_00.Model den MIRAS ALMAMALI"
# ======================================================================
# BURADAN ASAGISI MIMARI DEGIL. Uc AYRI kategori, karistirilmasin:
#
#   A) DILIN KENDISI (korpus)  -- hiperparametre DEGIL. "ek_kip=tr"
#      demek "bu dil ekli bir dil" demek; "bicim=3" demek "ayni olgu
#      uc yuzey biciminde gecıyor" demek. Modelin secimi degil,
#      METNIN ozelligi.
#   B) SINAV BOLMELERI          -- olcumun tanimi, modele ait DEGIL.
#   C) PROJEYE OZGU VERI EKI    -- BEST PRACTICE DEGIL. Tek kalem:
#      identity bridge (ident_frac / ident_kip). model_b13'te
#      arXiv 2509.24653'ten alindi. Burada DURUYOR cunku egitim havuzu
#      model_b15 ile BIT DUZEYINDE ayni olmali; olmazsa kol "mimari mi
#      veri mi" sorusunu AYIRAMAZ. Kaldirilirsa kol AYRI bir soru sorar
#      (onkayit model_00.md 2).
#
# Optimizasyon (wd/cosine/isinma/lr/betas) YUKARIDA degil ASAGIDA ve
# hepsi STANDART TARIF -- proje kisiti DEGIL.
# ======================================================================
# --- A) DILIN KENDISI -------------------------------------------------
assert M.AYAR.veri_ad == "veri_00",    "model_00 KENDI veri modulu"
import veri_00 as _V00
assert _V00.graf_izi(_V00.kur(0)) == _V00.IZ, "graf KAYMIS"
print(f"veri_00  graf izi {_V00.IZ}  (icerik veri_okul4, 1060 varlik)")
assert M.AYAR.jeton_ad == "tam",       "varlik = JETON DIZISI (3 yuva)"
assert M.AYAR.ek_kip == "tr",          "dil EKLI: '  <NIN>  <SI>  <DIR>"
assert M.AYAR.bicim == 3,              "ayni olgu UC yuzey biciminde"
assert M.AYAR.t_len == 17,             f"t_len (ek_kip'ten TURER): {M.AYAR.t_len}"
assert M.AYAR.belge_pay == 0.0,        "BELGE satiri YOK (ek_kip ile kurulmadi)"

# --- B) SINAV BOLMELERI -- olcumun tanimi -----------------------------
assert M.AYAR.ood_pay == 0.05,         f"ood bolmesi: {M.AYAR.ood_pay}"
assert M.AYAR.kopru_kayip == 0.0,      "YARDIMCI KAYIP YOK (bu MIMARI kararı)"

# --- C) PROJEYE OZGU VERI EKI -- BEST PRACTICE DEGIL ------------------
# Kullanici, 16 Eylul: "biz her seyi sifirdan yaptik, ben kisit
# vermedim, best practice dedim." Dogru -- ve bu IKI SATIR o tarifin
# parcasi DEGIL. Egitim havuzu b15 ile ayni kalsin diye duruyor.
assert M.AYAR.ident_frac == 0.2,       "identity bridge -- PROJE EKI, b15 ile AYNI"
assert M.AYAR.ident_kip == "q1",       "identity bridge -- PROJE EKI, b15 ile AYNI"

# --- OPTIMIZASYON: STANDART TARIFIN KENDISI ---------------------------
# Bunlar proje kisiti DEGIL. Dordu de ayni sayilari kullaniyor:
#   nanoGPT   GPT-3   Llama   Pythia
assert M.AYAR.wd == 0.1,               "wd 0.1 -- nanoGPT/Pythia/Qwen2.5 SFT"
assert M.AYAR.sabit_lr is False,       "cosine -> lr/10 -- nanoGPT/Pythia"
assert M.AYAR.lr == 1e-3,              "Pythia-70m ile ayni mertebe"
assert M.AYAR.isinma == 2000,          "nanoGPT warmup_iters=2000, Llama 2000"
assert M.AYAR.betas == (0.9, 0.95),    "nanoGPT/GPT-3/Llama/Pythia -- 0.999 DEGIL"
assert M.AYAR.tam_kayip is True,       "butun pozisyonlarda next-token: standart LM"
# PROJEYE OZGU OPTIMIZASYON NUMARALARI -- HEPSI KAPALI
assert M.AYAR.ort_bas == 0,            "LOOKAHEAD KAPALI -- hicbir tarifte YOK"
assert not (M.AYAR.dar_sert or M.AYAR.dar_sdpa), "darbogaz zaten YOK"
print("mimari: ModelSade  8 katman  RoPE  SwiGLU  bagli gomme  bias YOK")
SOR = os.path.join(AILE, "sor_00.py")  # 9. hucre kullanir
for _g in ("AYAR", "egit", "fark_bas"):
    assert hasattr(M, _g), f"{MODEL}'de {_g} YOK -- kos_00.py duser"
print("ayar:", M.AYAR)

## 4 — Başlat

**Önce tek tohum** (`t0`). Sonuç olumluysa yeter; olumsuzsa `TOHUMLAR`'ı
`[1, 2]` yapıp tekrar koşarsın — `t0` klasörüne dokunulmaz.

Çıktı Drive'da, **tepe klasör modelin adı**:

```
MyDrive/<model>/
    OKU.md        klasörü anlatır, her koşuda yenilenir
    log/          kos_<zaman>.txt
    t0/           ayar_ · kosu_ · egri_ · pencere_ json'ları
        snap/     snap_<model>_t0_00020000.pt
```

Dosya adları da tohumu taşır — klasörden çıkarmak gerekmiyor.

**Bütçeyi uzatmak** (CLAUDE.md kural 1): `ADIM`'ı büyüt **ve**
`SURDUR = True` yap. Uzatma sıfırdan koşu değil, **sürdürmedir** — koşu
kaldığı yerden devam eder. 20.000 bir tavan değil, ilk sınır.

> ⚠️ **Cosine + uzatma = LR YENİDEN BAŞLAR.** `model_b14`te ölçüldü:
> 20.000 ufkunda LR `lr/10`a inmişti; `ADIM`ı 60.000 yapınca 22.000'de
> LR 7,6 kat zıpladı ve kayıp geçici olarak yükseldi. Bozulma değil,
> **warm restart** — ama uzatılmış koşu temiz bir 60.000 koşusuyla
> **aynı şey değildir**.

> **tekrar:** ⛔ **hayır** (yeni koşu için). Üç koruma var: hücre, süreç
> hâlâ koşuyorsa ikincisini başlatmaz; `kos.py` dolu tohum klasörünü
> reddeder; sürdürme paketi yokken `SURDUR = True` koşuyu durdurur.
> `SURDUR = True` ile tekrar çalıştırmak ise **kasıtlı ve güvenlidir**.
> · **GPU:** evet (alt süreç; ölçüldü: model_a 20.000 adım **10,8 dk**,
> model_a2 **11,4 dk**, model_a1 **6,1 dk**)
> — hücre kendi içinde **GPU kapısını** sorar (kural 2)
> · **yazar:** Drive

**Ayrı süreç.** Çekirdek serbest kalır: ilerlemeye bakabilir, durdurabilirsin.

Koşan kod `deneme2/model_00/kos_00.py` — **depoda**, defterin içinde
metin olarak kurulmuyor. Yani başlatan kod da koşan kodla aynı commit'te.
Paylaşılan `kos.py` değil: `model_00` kendi koşucusunu taşıyor ve
`--model` seçeneği yok.

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# Sonuc OLUMLU cikarsa (ent yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
#
# Neden onemli: grokking tohuma bagli (2603.25009'da "only 1 of 3 seeds
# grokked" gibi sonuclar var). Tek tohumda ent yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

ADIM   = None    # None = ayar_00.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1). Yazili bir
#                  UST SINIR YOK; 40.000 da 60.000 da denebilir.
#                  UZATMA = SURDURME: asagidaki SURDUR'u da True yap.
#                  !! COSINE + UZATMA = LR YENIDEN BASLAR (model_b14'te
#                  olculdu: 22.000'de LR 7,6 kat zipladi).
SURDUR = False   # True = surdurme_t<N>.pt'den KALDIGI YERDEN devam.
#                  Paket YOKSA kosu DURUR ve bunu soyler -- sessizce bastan
#                  baslamaz. Surdurme destegi 15 Eylul'de eklendi; ondan
#                  onceki kosularda paket YOK.
USTUNE = False   # True = dolu klasoru t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

# KENDI kosucusu: model_00/kos_00.py. `--model` YOK -- bu betik
# yalnizca model_00'i baslatir (paylasilan kos.py aile klasoru ARIYORDU).
_arg = [sys.executable, "-u", f"{KOD}/deneme2/model_00/kos_00.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")

## 5 — İlerleme (ham log)

Alt sürecin çıktısının sonunu basar. **Hesap yapmaz, GPU kullanmaz** —
sadece dosya okur.

> ⚠️ **Log GERİDEN GELEBİLİR.** Alt sürecin `stdout`'u Drive'a yazılıyor ve
> FUSE katmanı onu tamponluyor. Ölçüldü (15 Eylül, canlı koşuda): log hâlâ
> adım 0'ı gösterirken 6. hücre adım 2000'i gösteriyordu. **Canlı durum için
> 6. hücreye bak**; bu hücre traceback okumak için.
>
> **tekrar:** güvenli, istediğin kadar · **GPU:** hayır · **yazar:** hayır
>
> Çekirdek yeniden başladıysa (`p` kaybolur) bu hücre yine çalışır: süreç
> tablosuna bakıp koşunun yaşayıp yaşamadığını söyler.

In [ ]:
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_00.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos_00.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

## 6 — Rapor (canlı durum)

Eğri ve künye dosyalarını Drive'dan okur; ikisi de **atomik** yazıldığı için
yarım dosya okunmaz ve **5. hücrenin aksine geriden gelmez**.

Koşu sürerken bakılacak hücre budur: hangi adımda, hangi kapı geçti,
`durum` ne.

> Yine de bu bir **ön okuma**. Birincil okuma `pencere_a.py` (7. hücre) —
> o, N anlık görüntünün **ağırlık ortalamasını** alıp tek model ölçer. Eğri
> değerleriyle pencere değerleri **aynı şey değildir**; arşivde ikisi ~2×
> farklı çıktı.
>
> **tekrar:** güvenli, istediğin kadar · **GPU:** hayır · **yazar:** hayır

In [ ]:
import json, glob, os, statistics

# KIYAS: model_b15 ILE AYNI VERI, AYNI SINAV (iz d6751004648c).
# Veri modulunun ADI farkli (veri_00), ICERIGI ayni -- kilit
# grafi, egitim havuzunu ve olcme izini ayri ayri sinadi.
# Degisen YALNIZ MIMARI. Bu, projedeki NADIR temiz kiyaslardan.
# !! AMA parametre IKI KATI (6,50M vs 3,23M) -- sonuc olumluysa
# "mimari mi parametre mi" AYRILAMAZ (onkayit model_00.md 3).
# model_b8 kopruyu EGITIMDE ZORLAYARAK koydu; bu veride nereye kadar
# gidilebildigini gosteriyor. model_b13 literaturun cozumunu deniyor:
# identity bridge (arXiv 2509.24653) -- "b -> b zero-hop sequences".
# Kendi KAHIN testimiz onlarin teshisini dogruladi: kopru elden
# verilse bile bilesim olmuyor ("contextual disconnect between the
# input and output spaces").
#
# !! ABLASYON DEGIL: model_b10'a gore UC dugme dondu (ident cifti +
# wd + cosine). Atfetme YAPILAMAZ; olumluysa bisect b13a/b13b.
#
# `kayip` sutunu artik DIL kaybi. model_b6 ile KIYASLANABILIR olan
# sutun `kayip_ana` (yalniz cevap yuvalari) -- ikisi de basiliyor.
# BIRINCIL olcu `comp`/`ent`. `ood` RAPORLANIR, HUKUM VERMEZ.
# !! BU SUTUNLAR BASKA BIR SINAV KUMESINDEN -- olcme izi 57cf60a5e9af.
# YON gostermek icin basiliyor, HUKUM icin DEGIL. model_b14 KENDI
# esiklerine gore okunur (onkayit model_b14.md 7).
# model_b15 BITTI (16 Eylul, durum BITTI, 20.000 adim). Asagidaki
# sayilar onun EGRISININ son olcumu -- `pencere_b` HENUZ KOSULMADI.
# !! EGRI ile PENCERE AYNI SEY DEGIL (CLAUDE.md kural 3 / birincil
# okuma): model_a'da ikisi 1,74x farkli cikmisti. Bu sutun YON
# gostergesi; b15'in HUKMU pencere_b ile verilecek ve o gelince
# buraya YAZILACAK.
B1 = dict(one=1.0000, seen=1.0000, comp=0.0580, ood=0.0143,
          ent=0.0250, ent_yok=0.0409)          # model_b15 EGRI, adim 20.000
B6 = dict(one=0.9347, seen=0.8757, comp=0.0657, ood=0.0133,
          ent=0.0273, ent_yok=0.0601)          # model_b6, BASKA sinav kumesi
B8 = dict(one=0.9510, seen=0.9913, comp=0.9747, ood=0.0177,
          ent=0.8650, ent_yok=0.8710)
B1_MS = 60.1                                 # model_b15 OLCULDU (t_len 17)

# `ent_yok` DOGRULUGU da olculuyordu ama tabloda YOKTU (15 Eylul): egri
# json'ina yaziliyor, rapora girmiyordu. Olculup gosterilmeyen sayi, yok
# sayilan sayidir -- eklendi.
# SUTUNLAR EGRININ KENDISINDEN TURETILIR, elle yazilmaz. `ent_kati`
# (kati_pay > 0) ancak boyle gorunur; sabit liste olsaydi olculur ama
# BASILMAZDI. Bu tam olarak `ent_yok`un basina gelen seydi (15 Eylul):
# egri json'ina yaziliyordu, rapora girmiyordu.
_TUM = (("adim", "adim"), ("kayip", "kayip"), ("one", "one"),
        ("seen", "seen"), ("comp", "comp"), ("ood", "ood"), ("ent", "ent"),
        ("ent_kati", "ent_kati"), ("ent_yok", "ent_yok"),
        ("kayip_ana", "kayip_ana"),
        ("ent_kisayol", "ent_ksy"), ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{os.path.basename(kl)}: egri YOK"); continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    SUT = tuple(x for x in _TUM if x[0] in _var)
    # 15 Eylul: `ood` SABIT listede unutulunca model_a8'de OLCULDU ama
    # BASILMADI. Artik atlanan sutun varsa hucre DUSER.
    _atlanan = [c for c in _var
                if c.startswith(("one", "seen", "comp", "ood", "ent"))
                and c not in {x[0] for x in SUT}]
    assert not _atlanan, f"OLCULUYOR AMA BASILMIYOR: {_atlanan}"
    _iz = k.get("olcme_izi", "?")
    assert _iz == "d6751004648c", (
        f"olcme izi {_iz} != d6751004648c -- veri_00 KIYASI GECERSIZ")
    print("=" * 84)
    print(f"{os.path.basename(kl)}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    v = k.get("veri", {})
    if v:
        print(f"   olgu {v['olgu']}  egitim2 {v['egitim2']}  ENT {v['ent']}  "
              f"phi {v['phi']} (wang {v['wang_phi']})  "
              f"parametre {k.get('parametre',0):,}  iz {k.get('olcme_izi','?')}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 81)
    for ad, kural, g in (
            ("SAGLIK-1HOP",  "one >= 0.98",  s.get("one", 0) >= 0.98),
            ("SAGLIK-EZBER", "seen >= 0.95", s.get("seen", 0) >= 0.95),
            ("OLGUNLUK",     "comp >= 0.50", s.get("comp", 0) >= 0.50),
            ("BIRIM TESTI",  "ent_yok_kisayol == 0",
             abs(s.get("ent_yok_kisayol", 1)) < 1e-9)):
        print(f"   {'GECTI ' if g else '!! KALDI'}  {ad:<14} {kural}")
    if len(e) >= 2:
        print(f"   ent son iki olcumde {e[-1].get('ent',0)-e[-2].get('ent',0):+.4f}"
              "   (BUTCE hukmu pencere_b'de verilir, burada DEGIL)")

    print("   " + "-" * 81)
    # HIZ: t_len 17, havuz 297.548 -- model_b15 ile AYNI. Adim basina
    # is FARKLI: parametre iki kati, yani daha yavas BEKLENIR.
    d = [(b["sn"] - a["sn"]) / (b["adim"] - a["adim"]) * 1000
         for a, b in zip(e, e[1:])
         if b["sn"] > a["sn"] and b["adim"] > a["adim"]]
    if d:
        ms = statistics.median(d)
        print(f"   HIZ ortanca {ms:.1f} ms/adim   (model_b15 {B1_MS})"
              f"   {(ms/B1_MS-1)*100:+.1f}%   [ayni t_len; parametre IKI KATI]")
    print(f"   {'olcu':<9}{'b6*':>10}{'b15 EGRI*':>11}{'model_00':>10}"
          f"{'b8 TAVAN':>10}")
    for c in ("one", "seen", "comp", "ood", "ent", "ent_yok"):
        if c in s:
            print(f"   {c:<9}{B6[c]:>10.4f}{B1[c]:>11.4f}{s[c]:>10.4f}"
                  f"{B8[c]:>10.4f}")
    print("   * b15 sutunu EGRI okumasi (adim 20.000); pencere_b HENUZ")
    print("     kosulmadi ve EGRI ile PENCERE ayni sey DEGIL (kural 3).")
    print("     b6 BASKA SINAV KUMESI. model_00 ile b15 AYNI sinavi")
    print("     goruyor (iz d6751004648c) -- kiyas o eksende GECERLI.")
    if "ood" in s:
        print(f"   ood {s['ood']:.4f} -- RAPORLANIR, HUKUM VERMEZ")
        print("       (onkayit model_00.md 6). 210 ornek, GURULTULU.")
    if "comp" in s:
        c_ = s["comp"]
        print("   BIRINCIL (comp -- KENDI esikleri, onkayit 7): "
              + (">=0.30 KLASIK MIMARI BILESIMI ACIYOR"
                 if c_ >= 0.30 else
                 "0.10-0.30 kismi -- yon DOGRU, miktar yetmiyor"
                 if c_ >= 0.10 else
                 "<0.10 mimari DEGIL -- kaynak baska yerde"))
        print("      ON KOSUL: one/seen kapilari. Gecmezse comp")
        print("      YORUMLANMAZ (onkayit model_00.md 5).")
        print("      !! PARAMETRE IKI KATI (6,50M vs 3,23M). Sonuc")
        print("      olumluysa 'mimari mi parametre mi' AYRILAMAZ;")
        print("      ayiran kol model_00s (d=192) olurdu.")
        print("      SINAV model_b15 ile AYNI (iz d6751004648c) --")
        print("      kiyas GECERLI, bu projede NADIR.")
        print("      b14 ile AYNI TABLODA okunamaz -- kiyas ARKA")
        print("      PLANDA (CLAUDE.md), kol IPTAL DEGIL.")
        print("      EK OKUMA: dogrusal sonda slot 0 -- Physics 3.1'in")
        print("      ASIL iddiasi orada (ezber != kodlama).")
    if "ent" in s:
        print(f"   UCUNCUL (ent {s['ent']:.4f}): RAPORLANIR, HUKUM VERMEZ --")
        print("      bolme jeton duzeyinde ZAYIF (293/312).")

## 7 — `pencere_<aile>` · BİRİNCİL OKUMA

Aile klasöründeki `pencere_*.py` (3. hücre bulur: `model_a` →
`pencere_a`, `model_b` → `pencere_b`).
N anlık görüntünün **ağırlık ortalamasını** alır, sonra **tek** model ölçer.
Eğri değerlerinin ortalaması değil — ikisi ayrı şey.

> **tekrar:** güvenli; dosya adı genişliği taşır
> (`pencere_<model>_t0_g5.json`), farklı genişlikler birbirini ezmez.
> · **GPU:** ⛔ **KOŞU SÜRERKEN ÇALIŞTIRMA.** Hücre kendi içinde
> `assert _bos > 2.0` soruyor; eğitim süreci belleği tuttuğu için bu
> kontrol DÜŞER ve hücre hata verir. Ölçüldü: kullanıcı 16 Eylül'de
> denedi, hata aldı. **Koşu BITTIKTEN sonra** çalıştır.
> · **yazar:** Drive (tohum klasörüne bir json)
> — hücre kendi içinde **GPU kapısını** sorar (kural 2)

### Eğitim koşarken hangi hücre çalıştırılabilir

| hücre | koşu sürerken | neden |
|---|---|---|
| 0 Ayarlar | ✅ | sadece değişken |
| 1 GPU | ✅ | sorar, kullanmaz |
| 2 Drive | ✅ | klasör açar |
| 3 Kod | ⛔ | `rm -rf /content/kod` — koşan süreç oradan import etti |
| 4 Başlat | ⛔ | hücre zaten engelliyor |
| 5 İlerleme | ✅ | dosya okur (geriden gelebilir) |
| 6 Rapor | ✅ | dosya okur, **güncel** |
| 7 `pencere_<aile>` | ⛔ | **GPU KAPISI DÜŞER** — eğitim belleği tutuyor, `_bos > 2.0` sağlanmıyor |
| 8 Durdur | ⛔ | koşuyu öldürür |

5 ve 6'nın güvenli olmasının sebebi teknik: anlık görüntü ve eğri **atomik**
yazılıyor (`.tmp` → `os.replace`), yani okuyucu ya eski ya yeni dosyayı
görür, arasını asla. Ölçüldü: atomik olmadan yarım bir `.pt`
`torch.load`'ı patlatıyor **ve** `pencere_a`'nın glob'una giriyordu.

**İKİNCİ DEFTER = İKİNCİ RUNTIME = İKİNCİ GPU.** Colab'da yeni defter açmak
aynı GPU'yu paylaşmaz. Orada sadece **2 → 3 → 7**'yi çalıştır, **4'e
dokunma**. Eğitim hiç etkilenmez; bedeli aynı anda iki Colab oturumu.

> **Ya da yerelde:** Drive bağlıysa bütün okumalar CPU'da yapılabilir —
> `pencere_00`, `asama1_00`, `tani_00`, `sor_00`. GPU'ya hiç dokunmaz.
> (`kahin_dok` bu kolda **yok**: Φ enjeksiyonunu ölçüyor, `model_00`'da
> Φ darboğazı kurulmuyor bile.)


In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

## 8 — Durdurmak, ve tekrar koşmak

`p.kill()` — ya da runtime'ı kapat. Anlık görüntüler Drive'da kalır,
`kosu_t<N>.json`'daki `durum` `HATA` olur ve rapor koşunun **bitmediğini**
söyler.

### Sürdürme VAR (15 Eylül'den beri)

Her ölçüm noktasında tek bir `surdurme_t<N>.pt` atomik olarak yazılır:
fp32 ağırlık **+ AdamW momentleri + GradScaler ölçeği + batch RNG durumu**.
Koşu koparsa en fazla `olc_her` adım kaybedilir.

Devam etmek ya da bütçeyi uzatmak için 4. hücrede:

```python
ADIM   = 40000     # yeni tavan
SURDUR = True      # kaldigi yerden
```

Sınandı: 8+8 adım sürdürülmüş koşu, kesintisiz 16 adımlık koşuyla **bit
düzeyinde aynı** çıktı (ağırlık farkı `0.000e+00`).

> ⚠️ **Ama LR programı için DEĞİL.** `sabit_lr=False` iken program
> `ayar.adim`dan türüyor; `ADIM`ı büyütmek ufku değiştirir ve LR geri
> yukarı fırlar. `model_b14`te ölçüldü: 20.000 ufkunda LR `lr/10`
> (1,0e-4) idi, 60.000 ufkunda 22.000. adımda **7,6e-4** oldu — 7,6 kat.
> Kayıp geçici yükseldi, sonra toparladı (warm restart gibi). Bozulma
> değil ama **uzatılmış koşu, temiz bir 60.000 koşusu DEĞİLDİR**. Anlık görüntüler bunu
yapamaz — içlerinde yalnız fp16 ağırlık var, optimizer durumu **yok**;
oradan devam etmek sessizce **başka bir yörünge** üretirdi.

> Sürdürme desteği eklenmeden **önce** koşulmuş klasörlerde paket yoktur.
> `SURDUR = True` o durumda koşuyu **durdurur** ve bunu söyler; sessizce
> baştan başlamaz.

### Tekrar koşunca ne kaybolur? — hiçbir şey

```
BASKA TOHUM (t1, t2)   ayri klasore yazar, t0'a DOKUNMAZ
AYNI TOHUM             kosu REDDEDILIR: "t0 ZATEN DOLU"
AYNI TOHUM + --ustune  eskisi SILINMEZ, t0_eski_<zaman>/ diye TASINIR
```

Üçü de ölçülerek sınandı. Eskiden `--ustune` klasörü temizlemeden üstüne
yazıyordu: 8 adımlık koşunun üstüne 4 adımlık koşu gelince `snap/` içinde
**iki koşunun anlık görüntüleri yan yana** kalıyordu ve `pencere_a` bunları
tek pencerede ortalıyordu — sessizce.

> **tekrar:** ⛔ koşuyu **öldürür**. Satırın başındaki `#` bilerek duruyor.
> · **GPU:** hayır · **yazar:** hayır

In [ ]:
# p.kill()

## 9 — `sor.py` · KENDİ SORUNU SOR

Kullanıcı isteği, 16 Eylül: *"ben kendim örneklerden soracağım, bir
input sonra da model bana cevap verecek. Mesela `Ayşe Yılmaz'ın annesi`
diye soracağım. Ya da `Ayşe Yılmaz'ın annesinin kardeşi`."*

**Artık Türkçe yazılıyor.** Dil ek işaretleyicili olduğu için soru da
öyle sorulur:

```
> Ayse Yilmaz'in annesi
  soru    Ayse Yilmaz'in annesi
  jeton   soru: Ayse | Yilmaz | <YOK>
  model   Feride Yilmaz
  gercek  Feride Yilmaz        DOGRU
  bolme   one   (atomik olgu -- egitimde GORULDU)

> Ayse Yilmaz'in annesinin kardesi
  soru    Ayse Yilmaz'in annesinin kardesi
  model   ...
  gercek  ...
  bolme   comp
  kopru   Feride Yilmaz   (model bunu YAZMADAN kullanmali)
```

**Türkçe harf şart değil.** `kardesi` de olur `kardeşi` de; `çocuğunun`
da olur `cocugunun` da. Çözümleyici 1161 yazımı tanıyor ve hiçbir ikisi
çakışmıyor (`_iliski_sozluk` bunu `assert` ile sınıyor — sessizce yanlış
ilişkiye bağlamak, hiç çözümlememekten kötüdür).

Çıplak yazım da **bozulmadı**: `Ayse Yilmaz anne kardes` yine çalışır.

**Neden gerçeği de basıyor:** graf sentetik. Model *"Feride Yilmaz"*
dediğinde doğru olup olmadığını bilemezsin. Araç modelin cevabını,
**graftaki gerçeği**, hangi **bölmeden** olduğunu ve varsa **köprüyü**
basar. Doğruluk **üç yuvanın da** tutmasını ister.

> ⚠️ **BU BİR ÖLÇÜ DEĞİL.** Elle sorulan sorular **seçilmiş**
> sorulardır; hüküm `pencere_00` ile verilir. Araç **anlamak** için.

> **tekrar:** güvenli · **GPU:** evet (tek ileri geçiş) · **yazar:** hayır

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK"
assert torch.cuda.mem_get_info()[0] / 1e9 > 2.0, "GPU dolu"
# ------------------------------------------------------------------------
# KENDI SORUNU SOR -- TURKCE. Listeye istedigin kadar satir ekle.
# Turkce harf SART DEGIL: "kardesi" de olur "kardesi" de (cozumleyici
# 1161 yazimi taniyor, hicbir ikisi cakismiyor).
# Ciplak yazim da calisir: "Ahmet Yilmaz anne kardes".
# !! BU BIR OLCU DEGIL -- elle sorulan sorular SECILMIS sorulardir.
SORULAR = [
    "Ahmet Yilmaz'in annesi",                    # 1-hop
    "Ahmet Yilmaz'in annesinin kardesi",         # 2-hop
    "Ahmet Yilmaz'in kardesinin okulu",          # 2-hop, cevap OKUL
    "Adana Lisesi'nin muduru",                   # 1-hop, OKUL -> KISI
    "Matematik'in hocasi",                       # 1-hop, DERS -> KISI
    "Adana'nin komsusunun valisi",               # 2-hop, SEHIR zinciri
]
_q = " ".join(f'--soru "{s}"' for s in SORULAR)
!python {SOR} {EV}/t{TOHUMLAR[0]} --genislik 5 {_q}

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# ASAMA-1 TESHISI -- model_00'da kopru hidden state'e girdi mi?
# !! BU KOLUN EK OKUMASI BURADA: Physics 3.1'in iddiasi
# 'augmentation olmadan bilgi EZBERLENIR ama DOGRUSAL KODLANMAZ'.
# Sonda slot 0 'bilgi VAR' derse comp acilmasa bile bu bir bulgu.
# arXiv 2505.17923 AYNI probe'u AYNI pozisyonda yapmis ve kopruyu
# BULMUS: 'the hidden representation of the last input token
# encodes information about all necessary bridge entities'.
# model_b13'te ayni pozisyonda 0.0275 cikmisti.
# MERDIVEN: model_b6 -> b9 -> model_b10 (TABAN) -> b13 -> b8.
#   model_b9 comp 0.0250 | model_b6 0.0442 | model_b8 TAVAN 0.9997
# --sonda : kopru DOGRUSAL okunabiliyor mu. Taban max(en_sik, KOPYA) --
#           kopru cogu zaman soru varligiyla ayni aileden, soyad GIRDIDE
#           duruyor (comp'ta kopya 0.4450). Bu duzeltilmeden slot1
#           yanlislikla "BILGI VAR" cikiyordu.

import os
ASAMA1 = os.path.join(AILE, "asama1_00.py")
assert os.path.exists(ASAMA1), ASAMA1
!python {ASAMA1} {EV}/t{TOHUMLAR[0]} --genislik 5 --sonda --birim --bolme comp,ent,ood,seen

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# AYRISTIRMA: model YANLIS cevap verirken NE diyor?
#   KISAYOL (r2'yi dogrudan soru varligina uygulamis)
#   KOPRU   (ara varligi yazmis, ikinci hop'u yapmamis)
#   VARLIK_DEGIL / ILGISIZ / ...
# Ayrica: 1.hop tek basina, 2.hop tek basina, IKISI BIRDEN -> KAYIP.
# Bu bir HUKUM olcusu DEGIL, arizanin SEKLINI gosterir.
import os
TANI = os.path.join(AILE, "tani_00.py")
assert os.path.exists(TANI), TANI
!python {TANI} {EV}/t{TOHUMLAR[0]} --genislik 5

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# TABAN KIYASI: model_00'in ariza SEKLINI model_b15 ile kiyasla.
# !! AYNI SINAV KUMESI (iz d6751004648c) -- sayilar DOGRUDAN
#    kiyaslanabilir. Degisen YALNIZ mimari.
# model_b15 bir ModelB oldugu icin onu `tani_b.py` ile okuyoruz.
# Bu kolun ariza SEKLI model_b15'ten FARKLI mi, yoksa ayni mi?
# Ayniysa "mimari hicbir seyi degistirmedi" DAHA GUCLU soylenir.
# EGITIM YOK, yalniz okuma -- model_b15/t0 Drive'da duruyor.
import os
TANI_B = "/content/kod/deneme2/model_b/tani_b.py"
EV_B6 = os.path.dirname(EV) + "/model_b15"
assert os.path.isdir(f"{EV_B6}/t0"), f"model_b15/t0 YOK: {EV_B6}"
!python {TANI_B} {EV_B6}/t0 --genislik 5